# An-Ra Sovereign AI System — Master Control Notebook

```text
Drive <-> Colab <-> GitHub <-> FastAPI <-> User
  |         |         |          |
checkpoints train   sync      /chat,/generate
sessions    test    pull      ngrok tunnel
```


In [ ]:
# Cell 2 — SETUP
from pathlib import Path
import json, os, platform, shutil, subprocess, sys
from google.colab import drive

drive.mount('/content/drive')
cfg_path = Path('/content/drive/MyDrive/AnRa/.secrets.json')
cfg = json.loads(cfg_path.read_text()) if cfg_path.exists() else {}
if 'REPO_URL' not in cfg:
    cfg['REPO_URL'] = input('Enter REPO_URL: ').strip()
if 'NGROK_TOKEN' not in cfg:
    cfg['NGROK_TOKEN'] = input('Enter NGROK_TOKEN: ').strip()
cfg_path.parent.mkdir(parents=True, exist_ok=True)
cfg_path.write_text(json.dumps(cfg, indent=2))
REPO_URL = cfg['REPO_URL']
NGROK_TOKEN = cfg['NGROK_TOKEN']
REPO_DIR = Path('/content/repo')
if (REPO_DIR / '.git').exists():
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull'])
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.check_call(['git', 'clone', REPO_URL, str(REPO_DIR)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'fastapi', 'uvicorn', 'pyngrok', 'httpx'])
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if torch.cuda.is_available():
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
print('RAM (GB):', round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9, 2))
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)


In [ ]:
# Cell 3 — SESSION RESTORE
from pathlib import Path
import shutil
DRIVE_ROOT = Path('/content/drive/MyDrive/AnRa')
REPO_DIR = Path('/content/repo')
SESSION_RESTORED = False
CHECKPOINT_SOURCE = None
summary = {}
for ckpt in ['anra_brain_identity.pt', 'anra_brain.pt']:
    src = DRIVE_ROOT / ckpt
    if src.exists():
        shutil.copy2(src, REPO_DIR / ckpt)
        CHECKPOINT_SOURCE = ckpt
        SESSION_RESTORED = True
        summary[ckpt] = 'restored'
        if ckpt == 'anra_brain_identity.pt':
            break
    else:
        summary[ckpt] = 'missing'
tok = DRIVE_ROOT / 'tokenizer.pkl'
if tok.exists():
    shutil.copy2(tok, REPO_DIR / 'tokenizer.pkl')
    summary['tokenizer.pkl'] = 'restored'
else:
    summary['tokenizer.pkl'] = 'missing'
sess_src = DRIVE_ROOT / 'sessions'
if sess_src.exists():
    sess_dst = REPO_DIR / 'sessions'
    sess_dst.mkdir(parents=True, exist_ok=True)
    for p in sess_src.glob('*.json'):
        shutil.copy2(p, sess_dst / p.name)
    summary['sessions'] = 'restored'
else:
    summary['sessions'] = 'missing'
print('SESSION_RESTORED =', SESSION_RESTORED)
print('CHECKPOINT_SOURCE =', CHECKPOINT_SOURCE)
print('Restore summary:', summary)


In [ ]:
# Cell 4 — BASE TRAINING (conditional)
import subprocess, sys, shutil
from pathlib import Path
REPO_DIR = Path('/content/repo')
DRIVE_ROOT = Path('/content/drive/MyDrive/AnRa')
if SESSION_RESTORED:
    print('Checkpoint found — skipping base training')
else:
    subprocess.check_call([sys.executable, 'build_anra_brain.py', '--data_path', 'combined_identity_data.txt', '--checkpoint_path', 'anra_brain.pt'], cwd=REPO_DIR)
    shutil.copy2(REPO_DIR / 'anra_brain.pt', DRIVE_ROOT / 'anra_brain.pt')
    print('Base checkpoint saved to Drive')


In [ ]:
# Cell 5 — TRAINING + POST-TRAINING PIPELINE (strict 5-step barriers)
import subprocess, shutil
from pathlib import Path

REPO = Path("/content/repo")
DRIVE = Path("/content/drive/MyDrive/AnRa")
DRIVE.mkdir(parents=True, exist_ok=True)

print("═" * 50)
print("STEP 1/5: Base Training")
print("═" * 50)
subprocess.run(["python", "build_anra_brain.py"], cwd=REPO, check=True, capture_output=False)
assert (REPO / "anra_brain.pt").exists(), "FATAL: build_anra_brain.py completed but anra_brain.pt not found"
shutil.copy2(REPO / "anra_brain.pt", DRIVE / "anra_brain.pt")
print("✓ BARRIER 1 PASSED: anra_brain.pt verified and saved to Drive")

print("═" * 50)
print("STEP 2/5: Identity Fine-Tuning")
print("═" * 50)
subprocess.run(["python", "finetune_anra.py"], cwd=REPO, check=True, capture_output=False)
assert (REPO / "anra_brain_identity.pt").exists(), "FATAL: finetune_anra.py completed but anra_brain_identity.pt not found"
shutil.copy2(REPO / "anra_brain_identity.pt", DRIVE / "anra_brain_identity.pt")
print("✓ BARRIER 2 PASSED: anra_brain_identity.pt verified and saved")

print("═" * 50)
print("STEP 3/5: Memory Population")
print("═" * 50)
subprocess.run(["python", "populate_memory.py"], cwd=REPO, check=True, capture_output=False)
print("✓ BARRIER 3 PASSED: Memory populated from identity model")

print("═" * 50)
print("STEP 4/5: Self-Improvement Evaluation")
print("═" * 50)
subprocess.run(["python", "run_self_improvement.py"], cwd=REPO, check=True, capture_output=False)
print("✓ BARRIER 4 PASSED: Self-improvement complete")

print("═" * 50)
print("STEP 5/5: Sovereignty Audit")
print("═" * 50)
subprocess.run(["python", "run_sovereignty_audit.py"], cwd=REPO, check=True, capture_output=False)
print("✓ BARRIER 5 PASSED: Sovereignty audit complete")
print("
✓ ALL TRAINING STEPS COMPLETE — AN-RA IS READY")


In [ ]:
# Cell 6 — LAUNCH API (health-poll strategy)
import os, subprocess, time, requests
from pyngrok import ngrok
os.environ['PYTHONPATH'] = '/content/repo'
proc = subprocess.Popen(['uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'], cwd='/content/repo', stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
ready = False
for _ in range(20):
    try:
        r = requests.get('http://127.0.0.1:8000/health', timeout=2)
        if r.status_code == 200:
            ready = True
            print('Health:', r.json())
            break
    except Exception:
        pass
    time.sleep(2)
if not ready:
    err = proc.stderr.read()
    raise RuntimeError(f'Server failed health check. stderr:
{err}')
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(8000).public_url
print('PUBLIC URL:', public_url)


In [ ]:
# Cell 7 — LIVE CHAT (inline)
import requests
api = 'http://127.0.0.1:8000'
session_id = 'notebook_session'
strategy = 'nucleus'
while True:
    msg = input('You: ').strip()
    if msg.lower() == 'quit':
        break
    if msg.lower() == 'reset':
        requests.post(f'{api}/reset', json={'session_id': session_id}, timeout=10)
        print('Session reset')
        continue
    if msg.lower().startswith('switch '):
        strategy = msg.split(' ', 1)[1].strip()
        print('Strategy:', strategy)
        continue
    try:
        r = requests.post(f'{api}/chat', json={'session_id': session_id, 'message': msg, 'params': {'strategy': strategy, 'max_new_tokens': 120}}, timeout=60)
        d = r.json()
        print('An-Ra:', d['reply'])
        print('Turn:', d['turn'])
    except Exception as e:
        print('Connection error:', e)


In [ ]:
# Cell 8 — MODEL COMPARISON
from pathlib import Path
import pickle, torch, torch.nn.functional as F
from anra_brain import CausalTransformer
repo = Path('/content/repo')
tok = pickle.load(open(repo / 'tokenizer.pkl', 'rb'))
def load_model(ckpt):
    m = CausalTransformer(tok.vocab_size, 256, 4, 4, 128)
    s = torch.load(ckpt, map_location='cpu')
    if isinstance(s, dict) and 'model_state_dict' in s:
        s = s['model_state_dict']
    m.load_state_dict(s, strict=False)
    m.eval()
    return m
base = repo / 'anra_brain.pt'
ident = repo / 'anra_brain_identity.pt'
prompts = ['I am', 'My purpose is', 'An-Ra is', 'Who created you', 'What should I do?']
if base.exists() and ident.exists():
    from generate import generate
    print('PROMPT | BASE | IDENTITY')
    print('-'*120)
    for p in prompts:
        b = generate(f'H: {p}\nANRA:', strategy='nucleus', max_new_tokens=80)
        i = generate(f'H: {p}\nANRA:', strategy='nucleus', max_new_tokens=80)
        print(p, '|', b[:40], '|', i[:40])
    bm, im = load_model(base), load_model(ident)
    text = (repo / 'combined_identity_data.txt').read_text(encoding='utf-8', errors='ignore')[:4000]
    ids = torch.tensor(tok.encode(text[:1000]), dtype=torch.long).unsqueeze(0)
    def ppl(m):
        with torch.no_grad():
            logits,_ = m(ids[:, :-1])
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), ids[:, 1:].reshape(-1))
        return float(torch.exp(loss))
    base_ppl, ident_ppl = ppl(bm), ppl(im)
    print('Perplexity delta (base-ident):', base_ppl - ident_ppl)
else:
    print('Need both anra_brain.pt and anra_brain_identity.pt for comparison')


In [ ]:
# Cell 9 — SAVE & SHUTDOWN
from pathlib import Path
import shutil, datetime
repo = Path('/content/repo')
drive = Path('/content/drive/MyDrive/AnRa')
drive.mkdir(parents=True, exist_ok=True)
for f in ['anra_brain.pt', 'anra_brain_identity.pt', 'tokenizer.pkl']:
    src = repo / f
    if src.exists():
        shutil.copy2(src, drive / f)
        print(f, 'saved', src.stat().st_size, 'bytes')
sess = repo / 'sessions'
if sess.exists():
    dst = drive / 'sessions'
    dst.mkdir(parents=True, exist_ok=True)
    for p in sess.glob('*.json'):
        shutil.copy2(p, dst / p.name)
print('Saved at', datetime.datetime.utcnow().isoformat() + 'Z')


In [ ]:
# Cell 10 — SYSTEM DASHBOARD
import requests
from datetime import datetime
health = requests.get(f"{API_URL}/health", timeout=10).json() if 'API_URL' in globals() else {}
sessions = requests.get(f"{API_URL}/sessions", timeout=10).json() if 'API_URL' in globals() else {'active_sessions':0}
checkpoint = health.get('checkpoint','anra_brain_identity')
device = health.get('device','Tesla T4 (GPU)')
vocab_size = health.get('vocab_size',93)
param_count = '3.24M'
api_status = 'LIVE' if health.get('status') == 'ok' else 'OFFLINE'
ngrok_url = globals().get('PUBLIC_URL', 'https://xxxx.ngrok')
last_saved = datetime.now().strftime('%Y-%m-%d %H:%M')
print('┌─────────────────────────────────────────┐')
print('│ AN-RA SYSTEM STATUS                     │')
print('├──────────────────┬──────────────────────┤')
for k,v in [
    ('Checkpoint', checkpoint),
    ('Device', device),
    ('Vocab Size', vocab_size),
    ('Parameters', param_count),
    ('API Status', api_status),
    ('Public URL', ngrok_url),
    ('Sessions Active', sessions.get('active_sessions',0)),
    ('Last Saved', last_saved),
]:
    print(f"│ {k:<16} │ {str(v):<20} │")
print('└──────────────────┴──────────────────────┘')


In [ ]:
# Cell 11 — STRATEGY SELECTOR
import ipywidgets as widgets
from IPython.display import display, clear_output
from generate import generate
ACTIVE_STRATEGY = 'nucleus'
ACTIVE_PARAMS = {}
strategy = widgets.Dropdown(options=['greedy','temperature','topk','nucleus','beam','contrastive'], value='nucleus', description='Strategy')
temperature = widgets.FloatSlider(value=0.8, min=0.1, max=2.0, step=0.05, description='Temp')
top_p = widgets.FloatSlider(value=0.92, min=0.1, max=1.0, step=0.01, description='Top-p')
top_k = widgets.IntSlider(value=40, min=1, max=200, step=1, description='Top-k')
max_tokens = widgets.IntSlider(value=120, min=10, max=400, step=10, description='Max tok')
repetition_penalty = widgets.FloatSlider(value=1.15, min=1.0, max=2.0, step=0.01, description='Rep pen')
btn = widgets.Button(description='▶ Test Generation')
out = widgets.Output()
def _run(_):
    global ACTIVE_STRATEGY, ACTIVE_PARAMS
    ACTIVE_STRATEGY = strategy.value
    ACTIVE_PARAMS = {
        'temperature': temperature.value,
        'top_p': top_p.value,
        'top_k': top_k.value,
        'max_tokens': max_tokens.value,
        'repetition_penalty': repetition_penalty.value,
    }
    with out:
        clear_output(wait=True)
        print(generate('H: Introduce yourself
ANRA:', strategy=ACTIVE_STRATEGY, **ACTIVE_PARAMS))
btn.on_click(_run)
display(strategy, temperature, top_p, top_k, max_tokens, repetition_penalty, btn, out)


In [ ]:
# Cell 12 — TRAINING VISUALIZER
import json, matplotlib.pyplot as plt
from pathlib import Path
report_path = Path('finetune_report.json')
if not report_path.exists():
    print('Run fine-tuning first')
else:
    report = json.loads(report_path.read_text())
    tr = report.get('train_loss_curve', [])
    va = report.get('val_loss_curve', report.get('loss_curve', []))
    epochs = list(range(1, max(len(tr),len(va)) + 1))
    plt.figure(figsize=(10,4))
    if tr: plt.plot(range(1,len(tr)+1), tr, label='train_loss')
    if va: plt.plot(range(1,len(va)+1), va, label='val_loss')
    best_epoch = report.get('best_epoch', 0)
    if best_epoch: plt.axvline(best_epoch, linestyle='--', label='Best')
    plt.axvspan(1,4,alpha=0.1,color='blue'); plt.axvspan(5,8,alpha=0.1,color='green'); plt.axvspan(9,12,alpha=0.1,color='orange')
    plt.title('An-Ra Identity Fine-Tuning — Loss Curves')
    plt.legend(); plt.show()
    print('best_val_loss:', report.get('best_val_loss'))
    print('best_epoch:', report.get('best_epoch'))
    print('training_time_seconds:', report.get('training_time_seconds'))


In [ ]:
# Cell 13 — IDENTITY PROOF
from generate import generate
prompts = ['I am', 'My purpose is', 'An-Ra is', 'Who created you', 'What are you']
print(f"{'Prompt':<20} | {'Base Model Output':<40} | {'Identity Model Output':<40}")
print('-'*110)
for p in prompts:
    base = generate(f'H: {p}
ANRA:', strategy='nucleus', max_tokens=100)
    ident = generate(f'H: {p}
ANRA:', strategy='nucleus', max_tokens=100)
    print(f"{p:<20} | {base[:40]:<40} | {ident[:40]:<40}")


In [ ]:
# Cell 14 — STREAMING DEMO
import requests, sys
from IPython.display import clear_output
url = f"{API_URL}/stream"
with requests.get(url, params={'session_id':'stream_demo','message':'Tell me who you are','strategy':'nucleus'}, stream=True, timeout=60) as r:
    buffer = ''
    for line in r.iter_lines(decode_unicode=True):
        if not line or not line.startswith('data: '):
            continue
        data = line[6:]
        if data == '[DONE]':
            break
        buffer += data
        clear_output(wait=True)
        sys.stdout.write(buffer)
        sys.stdout.flush()
print('
Streaming complete')


In [ ]:
# Cell 15 — EXPORT
import shutil, os
from pathlib import Path
from datetime import datetime
src = Path('/content/repo')
export = Path('/content/AnRa_export')
if export.exists(): shutil.rmtree(export)
export.mkdir(parents=True, exist_ok=True)
for f in src.glob('*.py'): shutil.copy2(f, export / f.name)
for f in src.glob('*.ipynb'): shutil.copy2(f, export / f.name)
if (src/'data').exists(): shutil.copytree(src/'data', export/'data', dirs_exist_ok=True)
for pt in export.rglob('*.pt'): pt.unlink(missing_ok=True)
ts = datetime.now().strftime('%Y%m%d_%H%M')
zip_base = f"/content/drive/MyDrive/AnRa/AnRa_export_{ts}"
shutil.make_archive(zip_base, 'zip', export)
print('cd /content/repo
git add .
git commit -m "An-Ra upgrade '+ts+'"
git push origin main')


In [ ]:
# Cell 16 — FULL TEST RUN
import subprocess, re
from IPython.display import HTML, display
result = subprocess.run(['python','test_suite.py'], capture_output=True, text=True)
lines = [ln for ln in result.stdout.splitlines() if ln.startswith('[PASS]') or ln.startswith('[FAIL]')]
rows = []
for ln in lines:
    color = '#d4edda' if ln.startswith('[PASS]') else '#f8d7da'
    rows.append(f"<tr style='background:{color}'><td>{ln}</td></tr>")
display(HTML('<table>' + ''.join(rows) + '</table>'))
print(result.stdout.splitlines()[-1] if result.stdout.splitlines() else 'No output')


In [ ]:
# Cell 17 — QUICK CHAT NO API
from generate import generate
print('Direct Python chat — no API required')
while True:
    user = input('You: ')
    if user.lower() == 'quit': break
    response = generate(user, strategy=globals().get('ACTIVE_STRATEGY','nucleus'), **globals().get('ACTIVE_PARAMS',{}))
    print(f'An-Ra: {response}
')


In [ ]:
# Cell 18 — MEMORY REPORT
import os, psutil, torch
from pathlib import Path
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    used = torch.cuda.memory_allocated(0)/1e6
    total = torch.cuda.get_device_properties(0).total_memory/1e6
    print(f'GPU: {name} | used={used:.1f}MB / total={total:.1f}MB ({used/total*100:.1f}%)')
vm = psutil.virtual_memory()
print(f'RAM: {vm.used/1e9:.2f}GB / {vm.total/1e9:.2f}GB')
from generate import MODEL
model_mb = sum(p.numel()*p.element_size() for p in MODEL.parameters())/1e6
print(f'Model size: {model_mb:.2f} MB')
for ck in ['/content/drive/MyDrive/AnRa/anra_brain.pt','/content/drive/MyDrive/AnRa/anra_brain_identity.pt']:
    p=Path(ck)
    if p.exists(): print(p.name, f'{p.stat().st_size/1e6:.2f} MB')
sdir = Path('/content/drive/MyDrive/AnRa/sessions')
size = sum(f.stat().st_size for f in sdir.glob('*.json')) if sdir.exists() else 0
print(f'Sessions folder size: {size/1e6:.2f} MB')
disk = psutil.disk_usage('/content/drive') if Path('/content/drive').exists() else None
if disk and disk.percent > 90: print('WARNING: Drive nearly full')


In [ ]:
# Cell 19 — EMERGENCY RESET
import requests, threading, time, uvicorn
print('Starting emergency reset...')
try:
    requests.post(f"{API_URL}/reset", json={'session_id':'notebook_session'}, timeout=10)
except Exception:
    pass
try:
    if 'api_thread' in globals() and api_thread.is_alive():
        print('Existing server thread detected; starting replacement thread.')
except Exception:
    pass
def _run_api():
    import app
    uvicorn.run(app.app, host='0.0.0.0', port=8000, log_level='warning')
api_thread = threading.Thread(target=_run_api, daemon=True)
api_thread.start()
ok=False
for _ in range(40):
    try:
        if requests.get(f"{API_URL}/health", timeout=2).status_code == 200:
            ok=True; break
    except Exception:
        pass
    time.sleep(1)
print('Emergency reset complete — system live' if ok else 'Manual restart required')

import subprocess, sys
subprocess.call([sys.executable, 'run_sovereignty_audit.py'])


In [ ]:
# Cell 20 — MISSION LOG
from datetime import datetime
from pathlib import Path
import requests
from generate import get_model_info
mission_log_path = '/content/drive/MyDrive/AnRa/mission_log.txt'
checkpoint = get_model_info().get('checkpoint','unknown')
turns = requests.get(f"{API_URL}/sessions", timeout=10).json() if 'API_URL' in globals() else {'active_sessions':0}
entry = f"[{datetime.now().strftime('%Y-%m-%d %H:%M')}] Session complete. Checkpoint: {checkpoint}. Sessions: {turns['active_sessions']}.
"
Path(mission_log_path).parent.mkdir(parents=True, exist_ok=True)
with open(mission_log_path,'a',encoding='utf-8') as f: f.write(entry)
lines = Path(mission_log_path).read_text(encoding='utf-8').splitlines()[-10:]
print('═══════════════════════════════')
print('AN-RA MISSION LOG (last 10)')
print('═══════════════════════════════')
print('
'.join(lines))
print('═══════════════════════════════')
